# Potato Leaf Disease Classification using Neural Network

In this notebook we classify potato leaf images as **Healthy** or **Diseased (Early Blight)** using a simple neural network. We start with a model that has no hidden layers, then add a hidden layer, and finally try a small CNN to see how performance improves.

This notebook follows the same structure as the `digits_recognition_neural_network` (MNIST) notebook, but adapted for real leaf images.

In [ ]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split

## 1. Load the dataset

We have two folders inside `data/`, next to this notebook:

- `data/Healthy/`  &rarr; healthy potato leaves (label `0`)
- `data/potato/`   &rarr; diseased leaves, Early Blight (label `1`)

We read every image, resize it to a fixed size, and stack them into a single numpy array `X` with labels `y` &mdash; exactly like `keras.datasets.mnist.load_data()` gives us `(X_train, y_train)` in the reference notebook.

In [ ]:
DATA_DIR = Path('./data').resolve()
IMG_SIZE = 64  # resize every leaf to 64x64 to keep training fast

class_dirs = {
    'Healthy': 0,
    'potato':  1,   # 'potato' folder contains Early Blight diseased leaves
}
class_names = ['Healthy', 'Diseased']

X, y = [], []
for folder, label in class_dirs.items():
    for img_path in sorted((DATA_DIR / folder).iterdir()):
        if img_path.suffix.lower() not in {'.jpg', '.jpeg', '.png'}:
            continue
        img = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        X.append(np.array(img))
        y.append(label)

X = np.array(X)
y = np.array(y)
print('X shape:', X.shape, '  y shape:', y.shape)
print('Healthy  :', (y == 0).sum())
print('Diseased :', (y == 1).sum())

## 2. Visualise a few samples

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, idx in zip(axes[0], np.random.choice(np.where(y == 0)[0], 4, replace=False)):
    ax.imshow(X[idx]); ax.set_title('Healthy'); ax.axis('off')
for ax, idx in zip(axes[1], np.random.choice(np.where(y == 1)[0], 4, replace=False)):
    ax.imshow(X[idx]); ax.set_title('Diseased'); ax.axis('off')
plt.tight_layout(); plt.show()

## 3. Normalize and split into train / test

Just like the MNIST notebook did `X_train = X_train / 255`, we scale pixel values to `[0, 1]`. Then we split 80/20 into train and test sets.

In [ ]:
X = X / 255.0

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train:', X_train.shape, y_train.shape)
print('Test :', X_test.shape,  y_test.shape)

## 4. Model 1 &mdash; Very simple NN, no hidden layers

Mirrors the first model in the MNIST notebook. We `Flatten` the image (64x64x3 = 12288 pixels) and feed it directly into a single output neuron with sigmoid activation (binary classification).

In [ ]:
model1 = keras.Sequential([
    keras.layers.Flatten(input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    keras.layers.Dense(1, activation='sigmoid'),
])

model1.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

model1.fit(X_train, y_train, epochs=5, validation_split=0.1)

In [ ]:
model1.evaluate(X_test, y_test)

## 5. Model 2 &mdash; With a hidden layer

Same as the second model in the MNIST notebook: add a hidden `Dense(128, relu)`. This usually improves accuracy a lot.

In [ ]:
model2 = keras.Sequential([
    keras.layers.Flatten(input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid'),
])

model2.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

model2.fit(X_train, y_train, epochs=10, validation_split=0.1)

In [ ]:
model2.evaluate(X_test, y_test)

## 6. Confusion matrix

Same idea as the MNIST notebook &mdash; build a confusion matrix and plot it as a heatmap. Since this is binary, the matrix is 2x2.

In [ ]:
import seaborn as sn

y_proba = model2.predict(X_test)
y_pred = (y_proba > 0.5).astype(int).flatten()

cm = tf.math.confusion_matrix(labels=y_test, predictions=y_pred)

plt.figure(figsize=(6, 4))
sn.heatmap(cm, annot=True, fmt='d',
           xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Truth')
plt.title('Confusion matrix &mdash; Dense NN')
plt.show()

## 7. Bonus &mdash; Small CNN

Dense networks throw away the 2D structure of the image. A **convolutional** network keeps it, and usually does much better on images. This mirrors "using Flatten layer" / "using hidden layer" progression in the MNIST notebook, one step further.

In [ ]:
cnn = keras.Sequential([
    keras.layers.Conv2D(16, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(32, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid'),
])

cnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

cnn.fit(X_train, y_train, epochs=10, validation_split=0.1)
cnn.evaluate(X_test, y_test)

In [ ]:
y_pred_cnn = (cnn.predict(X_test) > 0.5).astype(int).flatten()
cm_cnn = tf.math.confusion_matrix(labels=y_test, predictions=y_pred_cnn)

plt.figure(figsize=(6, 4))
sn.heatmap(cm_cnn, annot=True, fmt='d',
           xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Truth')
plt.title('Confusion matrix &mdash; CNN')
plt.show()

## 8. Predict on a single leaf

In [ ]:
idx = np.random.randint(len(X_test))
img = X_test[idx]
true_label = class_names[y_test[idx]]
pred_prob = cnn.predict(img[np.newaxis, ...])[0, 0]
pred_label = class_names[int(pred_prob > 0.5)]

plt.imshow(img)
plt.axis('off')
plt.title(f'Truth: {true_label}  |  Predicted: {pred_label}  (p={pred_prob:.2f})')
plt.show()